# 02 · Entrenamiento: la métrica, la validación, el control de fuga y la búsqueda

Notebook de la fase 2. Es narrativa: toda la lógica vive en `src/fraud_detection/` y aquí solo
se importa. Trabaja **solo con el periodo de referencia** (`step <= 400`).

Las cifras tienen dos orígenes, y cada una dice cuál:

- **Medidas en vivo**, por funciones de `src/` con tests: la regla, el piso del arnés, las
  curvas PR del fold 5, el mecanismo del control de fuga y el artefacto.
- **De registro**, transcritas de `docs/EVALUATION.md` y `docs/ERRORS_AND_LEARNINGS.md` con su
  identificador. Las evaluaciones por los dos splitters, los tests de permutación y la búsqueda
  costaron horas de cómputo y no se recalculan aquí. Cada transcripción se coteja en su celda:
  la media de los folds transcritos tiene que dar la media registrada, y donde una cifra se mide
  en vivo y además está registrada, la celda verifica que coincidan.

Las produjeron `scripts/run_default_models.py`, `scripts/run_leakage_check.py`,
`scripts/run_training.py` y `scripts/register_model.py`. Las decisiones y sus razones están en
los ADR 0007, 0009 y 0010.

In [ ]:
import os
import warnings

# Dos avisos de importación que no son del análisis e imprimirían rutas locales: la pista de
# MLflow, que importa `models.registry`, y la de tqdm, que importa optuna.
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fraud_detection.config import settings
from fraud_detection.data import profiling
from fraud_detection.features.builder import (
    DERIVATION_INPUTS,
    SIGNATURE_FEATURE,
    amount_equals_origin_balance,
)
from fraud_detection.models import estimators, search_space
from fraud_detection.models.baseline import BASELINE_TYPES, SignatureRule
from fraud_detection.models.evaluation import (
    DECISION_PRECISION,
    PRACTICAL_SIGNIFICANCE,
    ExpandingStepSplit,
    average_precision_by_block,
    cross_validate,
    features_and_labels,
    fold_table,
    load_evaluation_frame,
    ranking_head,
    recall_at_precision,
    score_fold,
    score_one_fold,
)
from fraud_detection.models.registry import (
    ARTIFACT_PATH,
    CHOSEN_PARAMS,
    CHOSEN_TRIAL,
    MODEL_INPUT_COLUMNS,
    chosen_model,
    load_pipeline,
)
from fraud_detection.models.tracking import file_sha256

pd.set_option("display.float_format", "{:.6g}".format)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 80
RULE, LR, RF, LGBM, CUT = "#dd8452", "#55a868", "#8172b2", "#4c72b0", "#c44e52"


def es(value: float, digits: int = 4) -> str:
    """Un número con coma decimal, para las anotaciones de las figuras."""
    return f"{value:.{digits}f}".replace(".", ",")

## 1. El problema de modelado

El problema no es "detectar fraude". En este dataset, una regla de una línea ya lo detecta casi
todo. La regla es la firma del simulador (ADR 0007): una transferencia o un retiro que se lleva
el saldo completo de la cuenta,

```
type ∈ {TRANSFER, CASH_OUT}  ∧  amount > 0  ∧  amount == oldbalanceOrg
```

No es fuga: el monto pedido y el saldo propio existen antes de la transacción. Es una firma
determinista del generador, cuyo agente de fraude toma una cuenta e intenta vaciarla.

In [ ]:
reference = load_evaluation_frame()
outcome = profiling.signature_rule_outcome(reference, BASELINE_TYPES)
frauds = outcome.true_positives + outcome.false_negatives
print(f"Referencia: {len(reference)} filas, {frauds} fraudes")
print(
    f"La regla marca {outcome.flagged_rows}: {outcome.true_positives} fraudes y "
    f"{outcome.false_positives} legítimas"
)
print(f"Precisión {outcome.precision:.3f}, recall {outcome.recall:.3f}")
print(f"Fraudes que la regla no cubre: {outcome.false_negatives}")

La regla marca 4.385 transacciones y las 4.385 son fraude: **precisión 1,000 y recall 0,979**.
Deja fuera 92 de los 4.477 fraudes de referencia.

Eso cambia la pregunta. Cualquier modelo que aprenda la firma se lleva esos 4.385 fraudes
gratis, y lo único que puede separar a un modelo de otro, y a un modelo de la regla, son **los
92 fraudes que la regla no cubre**. Todo el diseño de la evaluación sale de ahí.

## 2. La métrica de decisión, declarada antes de medir

El plan original usaba el PR-AUC como métrica principal: con 0,08% de positivos, el ROC-AUC es
engañosamente optimista. La celda siguiente puntúa a la regla como si fuera un modelo, con un
puntaje que vale 0 o 1, sobre toda la referencia.

In [ ]:
rows, labels = features_and_labels(reference)
rule_flags = SignatureRule().predict(rows)
rule_metrics = score_fold(labels, rule_flags.astype(float), rule_flags)
print(f"Prevalencia: {rule_metrics['prevalence']:.6f}")
print(f"PR-AUC de la regla, sin modelo: {rule_metrics['pr_auc']:.4f}")
print(f"Recall con precisión >= {DECISION_PRECISION}: {rule_metrics['recall_at_p0.99']:.4f}")
print(
    f"Significancia práctica: +{PRACTICAL_SIGNIFICANCE} de recall, "
    f"{PRACTICAL_SIGNIFICANCE * labels.sum():.1f} de los {labels.sum()} fraudes de referencia"
)

**El PR-AUC global está saturado por la firma.** Una regla sin modelo detrás ya saca 0,98, así
que cualquier modelo que aprenda la bandera empieza ahí y la métrica casi no tiene dónde
separarlos. Por eso la métrica de decisión es otra (ADR 0009):

- **Recall con precisión ≥ 0,99**: de todos los umbrales que mandan a revisión una cola donde a
  lo sumo 1 de cada 100 transacciones es legítima, el que más fraude atrapa. Es el punto de
  operación más allá de la regla: la regla tiene precisión 1,000, y un modelo vale lo que atrape
  de los 92 sin ensuciar la cola.
- **Secundaria:** recall con precisión ≥ 0,95, por si ningún modelo alcanzaba el piso de 0,99.
- **Contexto, siempre reportado:** PR-AUC, ROC-AUC y Brier.

**Se declaró antes de medir**, en el contrato de `docs/EVALUATION.md` y en el ADR 0009: una
métrica elegida después de ver resultados es la métrica que favorece el resultado que salió.
Con ella se declararon la política de empates (las filas con el mismo puntaje se marcan juntas,
porque ningún umbral las separa, y la precisión se compara en enteros) y el **umbral de
significancia práctica: +0,005 de recall**. Son unos 22 fraudes de referencia, unos 11 de las
cinco validaciones temporales y unos 2 por fold. Por debajo de eso, dos modelos son el mismo
modelo.

## 3. La validación cruzada temporal

Los folds se cortan por rangos de `step`, nunca dentro de uno, con **ventana expansiva**: el
fold *k* valida 40 steps y entrena con todos los anteriores. Es la situación de producción:
entrenar con toda la historia y predecir el periodo siguiente. `TimeSeriesSplit` no sirve,
porque parte por posición de fila y un borde puede caer dentro de una hora: el modelo
entrenaría con parte de un step y validaría con el resto. Producción no entra a ningún split:
el splitter rechaza una fila con `step > 400` en vez de ignorarla.

La celda siguiente corre el **predictor constante** por el arnés. Con todos los puntajes
empatados, el PR-AUC tiene que dar exactamente la prevalencia de cada fold: es el control del
instrumento antes de medir ningún modelo (EVAL-001).

In [ ]:
splitter = ExpandingStepSplit()
majority = cross_validate(
    reference, splitter, estimators.full_preprocessor, estimators.majority_class
)
folds = fold_table(majority, ("prevalence", "pr_auc", "baseline_recall"))
assert all(fold.metrics["pr_auc"] == fold.metrics["prevalence"] for fold in majority)
print("|PR-AUC - prevalencia| = 0 en los 5 folds: el arnés mide como se declaró")
folds.drop(columns="fit_seconds")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
for number, window in enumerate(splitter.validation_windows, start=1):
    ax.barh(number, window.first - 1, left=0.5, height=0.6, color="0.78")
    ax.barh(number, window.last - window.first + 1, left=window.first - 0.5, height=0.6, color=LGBM)
    row = folds.loc[number]
    validation = f"valida {row['validation_rows']:,} filas, {row['validation_positives']} fraudes"
    ax.text(8, number, validation.replace(",", "."), va="center", fontsize=9)
production = settings.REFERENCE_LAST_STEP + 0.5
ax.axvspan(production, 480, color=CUT, alpha=0.08)
ax.axvline(production, color=CUT, ls="--", lw=1.2)
ax.text(
    440,
    3,
    "step 401 en adelante:\nproducción,\nfuera de todo split",
    color=CUT,
    ha="center",
    va="center",
    fontsize=9,
)
ax.set_yticks(range(1, 6), [f"Fold {n}" for n in range(1, 6)])
ax.invert_yaxis()
ax.set_xlim(0, 480)
ax.set_xticks(range(0, 401, 40))
ax.set_xlabel("Step (1 step = 1 hora simulada)")
ax.set_title("Folds de ventana expansiva: gris entrena, azul valida")
plt.show()

**Figura 1.** Cada fold entrena con todo lo anterior a su ventana y valida 40 steps: de 1–200 →
201–240 a 1–360 → 361–400. Las ventanas de validación no se solapan, y juntas cubren los steps
201–400: 3.404.848 filas con 2.223 fraudes. Ninguna fila de producción entra a un fold. La
prevalencia de los folds va de 0,00051 a 0,00077, y el PR-AUC del constante la reproduce al
último dígito en los cinco: el arnés calcula lo que se declaró.

### 3.1 Los modelos con sus hiperparámetros por defecto

Las cifras de esta sección son de registro (EVAL-005 a EVAL-008 y EVAL-010): cada familia por
los dos splitters cuesta de segundos a media hora, y Random Forest sin acotar es la más cara. La
celda las transcribe y comprueba que la media de los cinco folds da la media registrada.

In [ ]:
# docs/EVALUATION.md: recall con precisión >= 0,99 por fold temporal, y media ± desviación
# estándar muestral (ddof = 1) registradas. El ensayo 5 es el modelo elegido (EVAL-010).
TEMPORAL = {
    "Regla (baseline)": ((0.9742, 0.9693, 0.9862, 0.9926, 0.9717), 0.9788, 0.0101, "EVAL-005"),
    "Regresión logística\npor defecto": (
        (0.9871, 0.9890, 0.9931, 0.9926, 0.9935),
        0.9911,
        0.0028,
        "EVAL-006",
    ),
    "Random Forest\npor defecto": ((0.9914, 1.0, 1.0, 1.0, 1.0), 0.9983, 0.0038, "EVAL-007"),
    "LightGBM\npor defecto": ((0.0, 0.0, 0.0, 0.0, 0.0), 0.0, 0.0, "EVAL-008"),
    "LightGBM elegido\n(ensayo 5)": (
        (0.9914, 0.9956, 1.0, 1.0, 0.9978),
        0.9970,
        0.0036,
        "EVAL-010",
    ),
}
for name, (values, mean, std, source) in TEMPORAL.items():
    assert abs(np.mean(values) - mean) <= 1e-4, (name, source)
    assert abs(np.std(values, ddof=1) - std) <= 1e-4, (name, source)
print("Transcripción cotejada: la media y la desviación de los folds dan las registradas")

In [ ]:
colors = [RULE, LR, RF, LGBM, LGBM]
offsets = np.linspace(-0.14, 0.14, 5)
rule_mean = TEMPORAL["Regla (baseline)"][1]
fig, ax = plt.subplots(figsize=(11, 5))
ax.axhspan(
    rule_mean,
    rule_mean + PRACTICAL_SIGNIFICANCE,
    color=RULE,
    alpha=0.13,
    label="Regla + 0,005: significancia práctica",
)
ax.axhline(rule_mean, color=RULE, ls="--", lw=1.2, label="Regla, media temporal (0,9788)")
for position, ((values, mean, std, _source), color) in enumerate(
    zip(TEMPORAL.values(), colors, strict=True)
):
    if mean == 0:
        ax.annotate(
            "0 en los 5 folds:\nno aprende (EVAL-008)",
            (position, 0.9535),
            xytext=(position, 0.962),
            ha="center",
            arrowprops={"arrowstyle": "->", "color": "0.3"},
        )
        continue
    ax.scatter(position + offsets, values, color="0.55", s=18, zorder=2)
    ax.errorbar(position, mean, yerr=std, fmt="o", color=color, ms=10, capsize=7, lw=2, zorder=3)
    ax.text(position + 0.2, mean, f"{es(mean)}\n± {es(std)}", va="center", fontsize=9)
ax.scatter([], [], color="0.55", s=18, label="Un fold")
ax.set_xticks(range(len(TEMPORAL)), list(TEMPORAL))
ax.set_ylim(0.95, 1.004)
ax.set_xlim(-0.5, len(TEMPORAL) - 0.3)
ax.set_ylabel("Recall con precisión ≥ 0,99")
ax.set_title("La métrica de decisión por modelo: 5 folds temporales, media ± desviación estándar")
ax.legend(loc="lower left", frameon=False)
plt.show()

**Figura 2.** La regla queda en 0,9788 ± 0,0101 y es la más dispersa: según el fold deja fuera
entre 3 y 14 fraudes. Con sus valores por defecto, la **regresión logística** llega a 0,9911 ±
0,0028 y **Random Forest** a 0,9983 ± 0,0038: los dos superan a la regla por más que la
significancia práctica. En diferencias pareadas, fold contra fold, son +0,0123 ± 0,0090 (28
fraudes más) y +0,0195 ± 0,0098 (44 más). Random Forest recupera todos los fraudes con precisión
≥ 0,99 en cuatro de los cinco folds. **LightGBM por defecto da 0 en los cinco**: su
entrenamiento diverge, y la sección 5 lo muestra. El modelo que terminó elegido, un LightGBM de
la búsqueda, queda en 0,9970 ± 0,0036, a la par de Random Forest.

### 3.2 Dónde queda la regla sobre cada curva: el fold 5

Ningún documento guarda curvas, así que esta se mide en vivo: tres modelos reajustados sobre un
fold por el mismo camino de código que el arnés (`score_one_fold`), y la curva leída por bloques
de puntaje (`average_precision_by_block`). El fold es el 5, que entrena con los steps 1–360 y
valida 361–400: el más parecido a producción y el de más historia. Se eligió por eso, no por su
resultado. Random Forest sin acotar tarda unos minutos.

In [ ]:
FOLD = 5
PR_MODELS = {
    "Regresión logística, por defecto": (estimators.logistic_regression, LR, 0.9935),
    "Random Forest, por defecto": (estimators.random_forest, RF, 1.0),
    f"LightGBM elegido (ensayo {CHOSEN_TRIAL})": (chosen_model, LGBM, 0.9978),
}
RULE_FOLD_5 = 0.9717  # EVAL-005, el fold 5 de la regla

curves, points = {}, {}
for name, (model, _, _) in PR_MODELS.items():
    scored = score_one_fold(reference, splitter, FOLD, estimators.full_preprocessor, model)
    curves[name] = average_precision_by_block(scored.labels, scored.scores)
    points[name] = recall_at_precision(scored.labels, scored.scores, DECISION_PRECISION)
    fold_labels, fold_rows = scored.labels, scored.validation
    del scored  # el modelo ajustado no se guarda: solo sus puntajes
fold_rule = SignatureRule().predict(fold_rows).astype(bool)
rule_recall = fold_labels[fold_rule].sum() / fold_labels.sum()

check = pd.DataFrame(
    {
        "recall en vivo": [point.recall for point in points.values()] + [rule_recall],
        "registrado": [recorded for _, _, recorded in PR_MODELS.values()] + [RULE_FOLD_5],
        "precisión en ese punto": [point.precision for point in points.values()]
        + [fold_labels[fold_rule].mean()],
        "fraudes atrapados": [point.true_positives for point in points.values()]
        + [int(fold_labels[fold_rule].sum())],
        "PR-AUC en vivo": [curve["contribution"].sum() for curve in curves.values()] + [np.nan],
    },
    index=[*PR_MODELS, "Regla (baseline)"],
)
assert (check["recall en vivo"] - check["registrado"]).abs().max() <= 5e-5
print(
    f"Fold {FOLD}: valida {len(fold_labels)} filas con {fold_labels.sum()} fraudes; "
    f"la regla deja fuera {int(fold_labels.sum() - fold_labels[fold_rule].sum())}"
)
check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, zoom in zip(axes, (False, True), strict=True):
    for name, (_, color, _) in PR_MODELS.items():
        curve = curves[name]
        ax.plot(
            curve["recall"],
            curve["precision"],
            drawstyle="steps-pre",
            color=color,
            lw=1.8,
            label=name,
        )
        if zoom:
            point = points[name]
            ax.plot(point.recall, point.precision, "o", color=color, ms=9, mec="white", zorder=4)
    ax.plot(
        rule_recall,
        1.0,
        "*",
        color=RULE,
        ms=17,
        mec="black",
        zorder=5,
        label="Regla (baseline): un solo punto",
    )
    ax.axhline(
        DECISION_PRECISION, color=CUT, ls="--", lw=1, label="Piso de la métrica de decisión, 0,99"
    )
    ax.set_xlabel("Recall")
    ax.grid(alpha=0.25, lw=0.5)
axes[0].set_xlim(0, 1.01)
axes[0].set_ylim(0, 1.02)
axes[0].set_ylabel("Precisión")
axes[0].set_title("Curva completa: las tres pegadas a la esquina")
axes[1].set_xlim(0.955, 1.002)
axes[1].set_ylim(0.90, 1.005)
axes[1].set_title("Acercamiento: el punto de cada modelo con precisión ≥ 0,99")
axes[1].legend(loc="lower left", frameon=False, fontsize=9)
fig.suptitle(f"Curvas PR en la validación del fold {FOLD} (steps 361 a 400)")
fig.tight_layout()
plt.show()

Los cuatro recalls medidos en vivo son los registrados en EVAL-005 a EVAL-010 para el fold 5,
y la celda se detendría si no lo fueran.

**Figura 3.** A la izquierda, la curva completa: las tres pegadas a la esquina superior derecha,
con PR-AUC entre 0,9975 y 1,0000 en este fold. Es la saturación de la sección 2 vista en la
curva: desde lejos, los tres modelos y la regla son el mismo.

A la derecha, el acercamiento. **La regla es un solo punto**, precisión 1,000 y recall 0,9717:
atrapa 447 de los 460 fraudes del fold. **Los tres modelos pasan por encima y a la derecha de ese
punto**: con la misma precisión, 1,000, llegan más lejos en recall, así que dominan a la regla.
Cada círculo es el punto que lee la métrica de decisión, el de mayor recall con precisión ≥ 0,99:
la regresión logística atrapa 457, el LightGBM elegido 459 y Random Forest los 460, los tres con
precisión 1,000 en ese punto. Pasado su punto, la curva de la regresión logística baja en
escalones: cada fraude más le cuesta legítimas en la cola.

Entre el mejor y el peor de los tres hay 3 fraudes. Es la escala a la que se mueve la métrica de
decisión en un fold, donde un fraude es 0,0022 de recall, y por eso la comparación entre modelos
se hace con la media de los cinco folds y su dispersión, no con un fold.

### 3.3 El contraste estratificado, y el resultado que contradice la intuición

`StratifiedKFold` mezcla las filas sin mirar el tiempo: valida con steps de los 400 y entrena
con filas del futuro de cada validación. Se mide al lado del temporal para cada modelo, no como
alternativa: es la medida del optimismo de ignorar el tiempo (ADR 0009). Lo esperable era que
diera **más alto**, porque ver el futuro solo puede ayudar. Las cifras son de registro (EVAL-005
a EVAL-007); el ensayo 5 no se midió estratificado, porque la búsqueda usó solo el splitter
temporal.

In [ ]:
# docs/EVALUATION.md: media ± desviación estándar entre 5 folds, temporal y estratificado.
CONTRAST = {
    "Regla (baseline)": ((0.9788, 0.0101), (0.9794, 0.0055), None, "EVAL-005"),
    "Regresión logística\npor defecto": ((0.9911, 0.0028), (0.9875, 0.0029), -0.0036, "EVAL-006"),
    "Random Forest\npor defecto": ((0.9983, 0.0038), (0.9931, 0.0018), -0.0052, "EVAL-007"),
}
fig, ax = plt.subplots(figsize=(10, 4.8))
for position, (temporal, stratified, difference, source) in enumerate(CONTRAST.values()):
    ax.errorbar(
        position - 0.12, temporal[0], yerr=temporal[1], fmt="o", color=LGBM, ms=9, capsize=6, lw=2
    )
    ax.errorbar(
        position + 0.12,
        stratified[0],
        yerr=stratified[1],
        fmt="s",
        color="0.45",
        ms=8,
        capsize=6,
        lw=2,
    )
    ax.plot(
        [position - 0.12, position + 0.12],
        [temporal[0], stratified[0]],
        color="0.6",
        lw=1,
        zorder=0,
    )
    if difference is not None:
        label = f"estratificado - temporal:\n{es(difference)} ({source})"
        ax.text(position + 0.2, stratified[0] - 0.0035, label, fontsize=9)
ax.errorbar([], [], yerr=[], fmt="o", color=LGBM, label="Temporal (decide)")
ax.errorbar([], [], yerr=[], fmt="s", color="0.45", label="Estratificado (contraste)")
ax.set_xticks(range(len(CONTRAST)), list(CONTRAST))
ax.set_xlim(-0.5, len(CONTRAST) - 0.2)
ax.set_ylim(0.96, 1.004)
ax.set_ylabel("Recall con precisión ≥ 0,99")
ax.set_title("Temporal contra estratificado, por modelo")
ax.legend(loc="lower right", frameon=False)
plt.show()

**Figura 4.** El estratificado no dio más alto: **dio más bajo** en los dos modelos que
aprenden, −0,0036 en la regresión logística y −0,0052 en Random Forest. En estos datos, ignorar
el tiempo no resultó optimista. LightGBM por defecto da 0 en los dos y no aparece. La regla da
casi lo mismo en los dos, porque no aprende nada de los datos de entrenamiento: su diferencia es
solo de qué filas cae en cada validación. Su desviación es casi la mitad en el estratificado,
donde cada fold tiene el doble de fraudes, unos 895; que esa sea la razón es una inferencia.

**La causa no está determinada.** Una diferencia de construcción que podría importar, sin medir
si importa: los dos splitters no validan sobre la misma población. El estratificado valida con
los 400 steps, incluidos los 1–200 que el temporal nunca valida, y entrena cada fold con el 80%
de toda la referencia, más que cualquier fold temporal salvo el último. Atribuir la diferencia a
eso sería una conjetura presentada como hallazgo.

Cronológicamente, el control de fuga de la sección siguiente corrió **antes** que estas
evaluaciones: la prueba del objetivo barajado se corre antes de comparar modelos. Se cuenta
después porque su lectura necesita estos resultados: un detector que no aprende con las
etiquetas reales no puede detectar una fuga.

## 4. El control de fuga

### 4.1 El test y la alarma

La defensa contra la fuga es la **prueba del objetivo barajado** (ADR 0009): en cada fold se
permutan las etiquetas **solo de la parte de entrenamiento**, la validación conserva las
verdaderas, y el Pipeline completo corre por el mismo arnés que los modelos. Un modelo que
aprendió de etiquetas al azar no puede ordenar la validación; si la ordena, algo filtra la
etiqueta a través de los folds o del preprocesamiento.

**La alarma.** La primera corrida, con una sola permutación, falló. El criterio pedía que el PR-AUC
colapsara a lo sumo a 10 veces la prevalencia de cada fold, y LightGBM en el fold 2 dio **14,83 veces**: un
PR-AUC de 0,0112 (EVAL-002). Quedó como bloqueante, y no se evaluó ningún modelo hasta
resolverlo.

**Resultó ser ruido.** El test se repitió como test de permutación: 20 permutaciones del mismo
modelo, con las semillas `settings.RANDOM_STATE + i`. La primera es la permutación original, y la
reprodujo al dígito: el 14,83 era el **máximo** de su distribución (EVAL-004). Dos errores de
diseño se juntaron:

- **Un cociente contra una prevalencia de 0,00076** convierte un ruido indistinguible de cero en
  una alarma de 14,83×. En absoluto, 0,0112 contra los ~0,98 de la firma con etiquetas reales.
- **Una permutación única es una muestra de tamaño uno**: no separa azar de fuga.

### 4.2 El nulo correcto

Para saber cuánto PR-AUC da la pura escasez de positivos, el **clasificador aleatorio**
(`UniformRandomScores`) puntúa cada fila con un uniforme, sin modelo, por el mismo arnés: 20
semillas × 5 folds (EVAL-003). Llega a 1,02 veces la prevalencia en la mediana y a 1,50 como
máximo. La escasez no explica un 14,83×.

Pero el aleatorio **no es el nulo del test**. El nulo de un test de objetivo barajado es **el
mismo modelo permutado**, nunca un ranking al azar. Acotan cosas distintas: el aleatorio acota
la inflación por pocos positivos; la permutación acota la inflación por **estructura**, la que
produce un modelo que agrupa filas por la geometría de las features aunque sus etiquetas sean
ruido. La que importaba era la segunda (ADR 0009, nota de estado; ERR-013).

El criterio que reemplazó al margen de 10×, declarado al cambiarlo y dicho así, porque se cambió
después de verlo fallar: el PR-AUC máximo de todas las permutaciones y folds de cada familia por
debajo de **0,05 en valor absoluto**, y la mediana del ROC-AUC dentro del rango del aleatorio o
por debajo. La figura usa los resúmenes registrados de cada distribución: los documentos guardan
mínimo, mediana, percentil 95 y máximo, y los 20 valores de LightGBM en el fold 2.

In [ ]:
# EVAL-003 y EVAL-004: PR-AUC bajo permutación (mínimo, mediana, p95, máximo) y valores.
LIGHTGBM_NULL = "LightGBM\npor defecto\npermutado, 20 x 5"
NULLS = {
    "Clasificador\naleatorio\n20 x 5 folds": (0.000491, 0.000736, 0.000825, 0.001135, "0.5"),
    "Regresión\nlogística\npermutada, 20 x 5": (0.000403, 0.001065, 0.002075, 0.004968, LR),
    LIGHTGBM_NULL: (0.000289, 0.000780, 0.003446, 0.011227, LGBM),
    "Random Forest\npermutado\n1 x 5": (0.000518, 0.000769, 0.000813, 0.000822, RF),
}
# fmt: off
LIGHTGBM_FOLD_2 = (
    0.011227, 0.000537, 0.000847, 0.000673, 0.000562, 0.003695, 0.000658, 0.000643, 0.000553,
    0.001100, 0.001456, 0.004532, 0.000532, 0.001008, 0.000975, 0.000787, 0.001198, 0.000568,
    0.002283, 0.000951,
)
# fmt: on
REAL_LABELS = {  # EVAL-005 a EVAL-007: PR-AUC temporal con etiquetas reales
    "Regla": (0.9788, RULE),
    "Regresión logística": (0.9967, LR),
    "Random Forest": (0.9983, RF),
}
FOLD_PREVALENCE = (0.000512, 0.000765)  # EVAL-001: rango de la prevalencia de los 5 folds
CRITERION = 0.05  # ADR 0009, nota de estado
assert len(LIGHTGBM_FOLD_2) == 20
assert max(LIGHTGBM_FOLD_2) == NULLS[LIGHTGBM_NULL][3]

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 6))
ax.axhspan(
    *FOLD_PREVALENCE, color="0.88", zorder=0, label="Prevalencia de los folds (0,00051 a 0,00077)"
)
ax.axhline(CRITERION, color=CUT, ls="--", lw=1.3, label="Criterio: máximo bajo permutación < 0,05")
for position, (low, median, p95, high, color) in enumerate(NULLS.values()):
    ax.plot([position, position], [low, high], color=color, lw=5, alpha=0.45, solid_capstyle="butt")
    ax.plot(position, median, "o", color=color, ms=10, mec="black")
    ax.plot(position, p95, "^", color=color, ms=9, mec="black")
    ax.plot(position, high, "v", color=color, ms=9, mec="black")
ax.scatter(
    np.full(20, 2.28),
    LIGHTGBM_FOLD_2,
    s=16,
    color=LGBM,
    alpha=0.7,
    label="LightGBM, fold 2: los 20 sorteos",
)
ax.annotate(
    "sorteo 0: 0,0112 = 14,83 veces la prevalencia\ndel fold 2, la alarma de EVAL-002",
    (2.28, 0.011227),
    xytext=(2.55, 0.0035),
    fontsize=9,
    arrowprops={"arrowstyle": "->", "color": "0.3"},
)
real_position = len(NULLS)
for offset, (name, (value, color)) in zip((-0.15, 0.0, 0.15), REAL_LABELS.items(), strict=True):
    ax.plot(
        real_position + offset,
        value,
        "D",
        color=color,
        ms=9,
        mec="black",
        label=f"{name}, etiquetas reales: {es(value)}",
    )
ax.plot([], [], "o", color="0.4", mec="black", label="Mediana")
ax.plot([], [], "^", color="0.4", mec="black", label="Percentil 95")
ax.plot([], [], "v", color="0.4", mec="black", label="Máximo (la barra va del mínimo al máximo)")
ax.set_yscale("log")
ax.set_ylim(1.5e-4, 2)
ax.set_xlim(-0.6, real_position + 0.6)
ax.set_xticks([*range(len(NULLS)), real_position], [*NULLS, "Con etiquetas\nreales"])
ax.set_ylabel("PR-AUC de validación (escala logarítmica)")
ax.set_title(
    "El test de permutación contra el clasificador aleatorio, y contra las etiquetas reales"
)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False, fontsize=9)
plt.show()

**Figura 5.** En escala logarítmica caben las tres cosas que importan. El **peor caso del nulo**
es el sorteo 0 de LightGBM, 0,0112, y las regresiones logísticas permutadas no pasan de 0,0050:
todo queda por debajo del criterio de 0,05. Las **etiquetas reales** dan de 0,98 a 0,998: la firma
sola, 0,98, ya queda unas 87 veces por encima del peor caso del nulo. Una fuga que recuperara la etiqueta dejaría el PR-AUC
en esa zona, no en la de abajo.

La cola de **LightGBM permutado es mucho más gorda que la del aleatorio**: su máximo es 9,9 veces
el del aleatorio (0,0112 contra 0,00113) y su percentil 95, 4,2 veces. Los 20 sorteos del fold 2
muestran que el 0,0112 es el único por encima de 0,005: la mayoría queda en la zona del
aleatorio. Si el nulo hubiera sido el aleatorio, el 14,83× se habría leído como fuga. Medido
contra el mismo modelo permutado, es el máximo de su propia distribución.

El ROC-AUC dice lo mismo desde otro lado (EVAL-004): una fuga lo empuja hacia 1, y bajo
permutación LightGBM queda en 0,242 de mediana, **por debajo** de 0,5: anti-generaliza, no
filtra. La regresión logística queda en 0,545, apenas por encima del rango del aleatorio
(0,470–0,539), y se aceptó con el mecanismo **sin determinar**.

**El detector que no aprende no detecta.** LightGBM con sus valores por defecto no aprende ni con
las etiquetas reales (figura 2), así que su paso por el test no tenía poder (EVAL-004, nota de
estado). La conclusión de que no hay fuga se sostiene en los dos detectores que sí aprenden: la
regresión logística, 0,0050 permutada contra 0,9967 real, y Random Forest, 0,00082 contra
0,9983.

### 4.3 El mecanismo, medido en vivo

La celda siguiente reajusta exactamente la permutación de la alarma: LightGBM por defecto, fold
2, etiquetas de entrenamiento permutadas con `settings.RANDOM_STATE`. Tiene que dar 0,011227402,
el valor registrado: es el control de determinismo del test. Después mira qué hay en la cabeza
del ranking (`ranking_head`) y de qué bloques de puntaje sale el PR-AUC
(`average_precision_by_block`), las dos mediciones de ERR-013.

In [ ]:
permuted = score_one_fold(
    reference,
    splitter,
    2,
    estimators.full_preprocessor,
    estimators.lightgbm_classifier,
    shuffle_training_labels=True,
    shuffle_seed=settings.RANDOM_STATE,
)
blocks = average_precision_by_block(permuted.labels, permuted.scores)
average_precision = blocks["contribution"].sum()
prevalence = permuted.labels.mean()
assert round(average_precision, 9) == 0.011227402  # ADR 0009, nota de estado: el sorteo 0
print(
    f"PR-AUC {average_precision:.9f} = {average_precision / prevalence:.2f} veces la "
    f"prevalencia del fold ({prevalence:.6f})"
)
signature = amount_equals_origin_balance(
    permuted.validation[list(DERIVATION_INPUTS[SIGNATURE_FEATURE])]
).to_numpy()
head = pd.DataFrame(
    ranking_head(permuted.labels, permuted.scores, signature, (10, 50, 100, 1000))
).set_index("k")
head.rename(
    columns={"flagged": "marcadas", "positives": "fraudes", "marked_positives": "fraudes con firma"}
)

In [ ]:
blocks["share"] = blocks["contribution"] / average_precision
top = blocks.sort_values("share", ascending=False).head(5)
holding_fraud = int((blocks["positives"] > 0).sum())
print(f"Bloques de puntaje empatado: {len(blocks)}; con algún fraude: {holding_fraud}")
print(f"Los tres que más aportan suman el {top['share'].head(3).sum():.1%} del PR-AUC")
top[["share", "rows", "positives", "flagged", "precision", "recall"]].rename(
    columns={
        "share": "parte del PR-AUC",
        "rows": "filas del bloque",
        "positives": "fraudes del bloque",
        "flagged": "puesto del último",
        "precision": "precisión",
    }
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(blocks["recall"], blocks["precision"], drawstyle="steps-pre", color=LGBM, lw=1.2)
big = blocks[blocks["share"] >= 0.02]
ax.scatter(
    big["recall"],
    big["precision"],
    s=big["rows"] * 6,
    color=CUT,
    alpha=0.6,
    zorder=3,
    label="Bloque que aporta ≥ 2% del PR-AUC (área ∝ filas)",
)
label_positions = ((0.36, 0.30), (0.50, 0.075), (0.36, 0.012))
largest = big.sort_values("share", ascending=False).head(3)
for (_, block), position in zip(largest.iterrows(), label_positions, strict=True):
    label = f"{int(block['rows'])} filas, {int(block['positives'])} fraudes, "
    ax.annotate(
        f"{label}puesto {int(block['flagged'])}\n{block['share']:.1%} del PR-AUC".replace(".", ","),
        (block["recall"], block["precision"]),
        xytext=position,
        fontsize=9,
        arrowprops={"arrowstyle": "->", "color": "0.3"},
    )
ax.axhline(prevalence, color="0.4", ls=":", lw=1, label="Prevalencia del fold 2")
ax.set_yscale("log")
ax.set_ylim(prevalence / 5, 1.2)
ax.set_xlim(0, 1.01)
ax.set_xlabel("Recall")
ax.set_ylabel("Precisión (escala logarítmica)")
ax.set_title("LightGBM con etiquetas de entrenamiento permutadas, fold 2: de dónde sale el PR-AUC")
ax.legend(loc="upper right", frameon=False)
plt.show()

**Figura 6.** El sorteo 0 se reprodujo al dígito, 0,011227402, y con él la alarma: 14,83 veces la
prevalencia del fold. Lo que hay detrás:

- **La cabeza del ranking no tiene fraudes.** Las 100 transacciones con mayor puntaje son
  legítimas, y entre las primeras 1.000 hay 7 fraudes. El PR-AUC no se infla por precisión alta
  arriba, que era la hipótesis inicial.
- **Se infla por saltos grandes de recall con precisión moderada.** Tres bloques de filas con
  exactamente el mismo puntaje —50, 20 y 7 filas, todas fraude— terminan en los puestos 1.496,
  1.552 y 1.518, con precisión de 5 a 7%, y aportan el 86,6% del PR-AUC. El de 50 aporta más de
  la mitad él solo: suma 11% de recall de golpe (ERR-013).
- **Pasado ese tramo, el modelo ordena al revés.** Desde un recall de 0,24 la precisión cae por
  debajo de la prevalencia: el resto de los fraudes queda debajo de las legítimas, y el ROC-AUC de
  este fold es 0,26 (EVAL-002).

**Por qué existen los bloques** (ERR-013). Un modelo de árboles corta sobre la bandera de la firma
aunque sus etiquetas sean ruido: elige el corte por impureza sobre las etiquetas de entrenamiento,
y un grupo raro y puro se ve informativo por azar. Puntaje idéntico es camino idéntico por todos
los árboles. Las filas de los bloques son fraudes con la firma, y son fraude en validación por la
relación real de la firma, que el modelo **no** aprendió: solo agrupó por ella.

La consecuencia es general: el nulo de permutación de este dataset tiene una cola unas 10 veces
más gorda que la de un ranking al azar, y la engorda la geometría de una feature, no información
de la etiqueta. El test sigue discriminando porque las etiquetas reales dan ~0,98, unas 87 veces
por encima del peor caso del nulo.

## 5. La búsqueda de hiperparámetros

El algoritmo es **un hiperparámetro más** (Reto ML 1): cada ensayo de Optuna elige primero la
familia —regresión logística, Random Forest o LightGBM— y después sus hiperparámetros, y se
puntúa con la métrica de decisión fold por fold sobre el splitter temporal. Un podador de mediana
corta un ensayo que ya va perdiendo. Los ensayos 0 a 2 son los **puntos de partida**: los valores
por defecto donde caben en el espacio y la cota más permisiva donde no. El espacio vive en
`models/search_space.py`; la tabla siguiente lo lee de ahí.

In [ ]:
space = [
    {"hiperparámetro": name, "dominio": ", ".join(str(choice) for choice in choices)}
    for name, choices in search_space.CHOICES.items()
] + [
    {"hiperparámetro": name, "dominio": f"{low} a {high}" + (", escala log" if log else "")}
    for name, (low, high, log) in {**search_space.INTS, **search_space.FLOATS}.items()
]
pd.DataFrame(space).set_index("hiperparámetro")

**El presupuesto era 4 horas o 150 ensayos. La búsqueda se cortó a los 8 ensayos terminados, unos
38 minutos, por presión de memoria**: el entorno mató el proceso cuando la memoria disponible bajó
a 231 MB (ERR-017). El ensayo 8 quedó a medio ajustar y se marcó como fallido. El resultado es el
de una búsqueda incompleta, y así está escrito (ADR 0010). Las cifras son de registro (EVAL-009).

In [ ]:
# EVAL-009: media ± desviación entre 5 folds y fit total de cada ensayo, en segundos. El ensayo 7
# fue podado tras el fold 1; su cifra es ese fold. El ensayo 2 diverge: 0 en los 5.
TRIALS = pd.DataFrame(
    [
        (0, "Regresión logística", 0.9911, 0.0028, 23, "punto de partida"),
        (1, "Random Forest", 0.9974, 0.0039, 625, "punto de partida"),
        (2, "LightGBM", 0.0, np.nan, 42, "punto de partida, diverge"),
        (3, "Random Forest", 0.9978, 0.0037, 960, ""),
        (4, "Regresión logística", 0.9859, 0.0091, 26, ""),
        (5, "LightGBM", 0.9970, 0.0036, 59, "elegido"),
        (6, "LightGBM", 0.9970, 0.0036, 72, ""),
        (7, "Regresión logística", 0.9763, np.nan, 4, "podado: solo el fold 1"),
    ],
    columns=["ensayo", "familia", "recall", "desviación", "fit total (s)", "nota"],
).set_index("ensayo")
complete = TRIALS[TRIALS["desviación"].notna()]
best = complete["recall"].max()
tied = complete.index[complete["recall"] >= best - PRACTICAL_SIGNIFICANCE].tolist()
assert tied == [1, 3, 5, 6]  # EVAL-009: cuatro configuraciones empatadas
cheapest = complete.loc[tied, "fit total (s)"].idxmin()
assert cheapest == CHOSEN_TRIAL
print(f"Mejor punto: ensayo {complete['recall'].idxmax()}, {best}")
print(f"Empatan a menos de {PRACTICAL_SIGNIFICANCE} del mejor: {tied}")
print(f"El de menor costo de ajuste entre los empatados: el ensayo {cheapest}")
TRIALS

In [ ]:
family_colors = {"Regresión logística": LR, "Random Forest": RF, "LightGBM": LGBM}
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.axhspan(
    best - PRACTICAL_SIGNIFICANCE,
    best,
    color=LGBM,
    alpha=0.1,
    label="Empate: a menos de 0,005 del mejor",
)
ax.axhline(rule_mean, color=RULE, ls="--", lw=1.2, label="Regla, media temporal (0,9788)")
for trial, row in TRIALS.iterrows():
    color = family_colors[row["familia"]]
    if row["recall"] == 0:
        ax.annotate(
            f"ensayo {trial}: 0 en los 5 folds\n(diverge), {row['fit total (s)']} s",
            (row["fit total (s)"], 0.9705),
            xytext=(row["fit total (s)"] * 1.6, 0.9735),
            fontsize=9,
            arrowprops={"arrowstyle": "->", "color": color},
        )
        continue
    pruned = np.isnan(row["desviación"])
    marker = "*" if trial == CHOSEN_TRIAL else "o"
    ax.errorbar(
        row["fit total (s)"],
        row["recall"],
        yerr=0 if pruned else row["desviación"],
        fmt=marker,
        ms=17 if trial == CHOSEN_TRIAL else 9,
        color=color,
        mfc="white" if pruned else color,
        mec=color,
        capsize=5,
        alpha=0.95,
    )
    if trial == CHOSEN_TRIAL:
        ax.text(
            row["fit total (s)"] * 0.88,
            row["recall"],
            f"elegido: {trial}",
            ha="right",
            va="center",
            fontsize=10,
        )
    else:
        note = " (podado: fold 1)" if pruned else ""
        ax.text(row["fit total (s)"] * 1.08, row["recall"] + 0.0006, f"{trial}{note}", fontsize=10)
for family, color in family_colors.items():
    ax.plot([], [], "o", color=color, label=family)
ax.set_xscale("log")
ax.set_xlim(2.5, 2500)
ax.set_ylim(0.969, 1.003)
ax.set_xlabel("Costo de ajuste: fit total de los 5 folds, segundos (escala logarítmica)")
ax.set_ylabel("Recall con precisión ≥ 0,99")
ax.set_title("Los 8 ensayos de la búsqueda: métrica contra costo")
ax.legend(loc="lower right", frameon=False, fontsize=9)
plt.show()

**Figura 7.** El punto más alto es el ensayo 3, un Random Forest con 0,9978, y **cuatro
configuraciones quedan dentro de la banda de empate**: los Random Forest 1 y 3 y los LightGBM 5 y
6. Dentro de la banda no hay ganador por métrica: el rango usable va de 0,9788 (la regla) a 1, y
con pasos de 0,005 caben a lo sumo cuatro niveles. Coronar un ganador por el tercer decimal de una
media de cinco folds sería ruido presentado como resultado.

**La regla de desempate se escribió antes de leer un solo resultado de la búsqueda**: la declaró el
Arquitecto el 2026-10-19 a las 08:09, con la búsqueda en curso (`docs/EVALUATION.md`). En orden:
el costo de ajuste medido, la simplicidad de la configuración y la dispersión entre folds. Por el
primer criterio **gana el ensayo 5**: 59 s contra 625 y 960 s de los Random Forest, unas diez
veces menos. Los costos se midieron con swap y están inflados, sobre todo los de Random Forest;
que el orden entre familias resista esa inflación es una inferencia, no una medición.

Los dos resultados negativos son resultados:

- **Regresión logística: la búsqueda no mejoró el punto de partida.** El ensayo 0 coincide con el
  valor por defecto real (0,9911), y los ensayos 4 y 7, con pesos de clase, quedaron por debajo.
- **Random Forest: la búsqueda empató con el valor por defecto.** El ensayo 3 da 0,9978, contra
  0,9983 del Random Forest por defecto sin acotar y 0,9974 de su punto de partida acotado. Acotar
  Random Forest a una configuración desplegable cuesta 0,0009, por debajo del umbral.

El resultado positivo es el de LightGBM: pasó de **divergir** (ensayo 2, 0) a 0,9970. Qué
parámetro contiene la divergencia no está determinado: los ensayos 5 y 6 subieron
`min_child_samples` a ~250 **y** bajaron la tasa de aprendizaje a la vez. Y con 8 de 150 ensayos,
nada de esto permite concluir que una búsqueda completa no encontraría algo mejor.

### 5.1 Por qué LightGBM por defecto da 0

La figura siguiente es una **medición ad hoc, no versionada** (ERR-014): un solo ajuste de LightGBM
con sus valores por defecto sobre el fold 5 temporal, con etiquetas reales, evaluado en cinco
iteraciones. Sus cifras están en el registro de errores, que dice que no sostienen ningún
documento entregable hasta que la medición se versione; aquí ilustran el mecanismo, y ninguna
conclusión de este notebook depende de ellas.

In [ ]:
# ERR-014, medición ad hoc: un ajuste sobre el fold 5, evaluado en estas iteraciones.
DIVERGENCE = pd.DataFrame(
    {
        "PR-AUC de entrenamiento": (0.9956, 0.9895, 0.9668, 0.4443, 0.0008),
        "PR-AUC de validación": (0.9978, 0.9956, 0.9701, 0.3866, 0.0008),
        "puntaje crudo máximo, en valor absoluto": (122, 1_285, 17_484, 79_355, 31_000_000),
    },
    index=pd.Index((1, 5, 20, 50, 100), name="iteración"),
)
fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(
    DIVERGENCE.index,
    DIVERGENCE["PR-AUC de entrenamiento"],
    "o--",
    color="0.45",
    lw=1,
    ms=8,
    label="PR-AUC de entrenamiento",
)
ax.plot(
    DIVERGENCE.index,
    DIVERGENCE["PR-AUC de validación"],
    "s--",
    color=LGBM,
    lw=1,
    ms=8,
    label="PR-AUC de validación",
)
ax.set_ylim(-0.03, 1.05)
ax.set_xscale("log")
ax.set_xticks(DIVERGENCE.index, [str(i) for i in DIVERGENCE.index])
ax.set_xlabel(
    "Iteración de boosting (escala logarítmica; las líneas solo unen los cinco puntos medidos)"
)
ax.set_ylabel("PR-AUC")
raw = ax.twinx()
raw.plot(
    DIVERGENCE.index,
    DIVERGENCE["puntaje crudo máximo, en valor absoluto"],
    "^-",
    color=CUT,
    ms=8,
    label="Puntaje crudo máximo en validación, |log-odds|",
)
raw.set_yscale("log")
raw.set_ylabel("|puntaje crudo| máximo (escala logarítmica)", color=CUT)
handles = ax.get_legend_handles_labels()[0] + raw.get_legend_handles_labels()[0]
ax.legend(handles, [h.get_label() for h in handles], loc="center left", frameon=False)
ax.set_title("MEDICIÓN AD HOC, NO VERSIONADA (ERR-014): LightGBM por defecto diverge")
plt.show()

**Figura 8.** En la primera iteración el modelo ya ordena casi perfecto: PR-AUC 0,998 en
validación. Después los puntajes crudos crecen sin cota, de 122 a 31 millones de log-odds, y el
PR-AUC colapsa **también en entrenamiento**, hasta 0,0008 en la iteración 100. Por eso no es
sobreajuste, que bajaría la validación y no el entrenamiento, y no es el arnés midiendo mal: es el
entrenamiento que se va. Por qué los valores por defecto divergen en estos datos —pasos de Newton
sin cota sobre una clase rara y casi separable— es una hipótesis no medida.

Su consecuencia está en la sección 4: un modelo que no aprende la etiqueta verdadera tampoco
aprendería una filtrada, y su paso por el test de permutación no prueba nada.

## 6. El modelo elegido

Es el ensayo 5 de la búsqueda (ADR 0010): un LightGBM con estos hiperparámetros, que
`models/registry.py` guarda con precisión completa porque el estudio de Optuna no está versionado y
un clon limpio tiene que reconstruir el mismo modelo.

In [ ]:
pd.Series(dict(CHOSEN_PARAMS), name=f"ensayo {CHOSEN_TRIAL}").to_frame()

**Su robustez ante la semilla.** La configuración está cerca de la región donde su familia diverge,
así que antes de serializarla se reajustó con otras tres semillas, 43, 44 y 45, sobre los cinco
folds temporales, con un criterio fijado antes de medir: cada media dentro de la dispersión del
ensayo 5, ±0,0036. Las tres dieron **el mismo recall por fold** que el ensayo 5 —0,9914 · 0,9956 ·
1,0 · 1,0 · 0,9978—, con distancia 0,0000 (EVAL-010). La medición podía fallar: la semilla sí
cambió el modelo, porque el PR-AUC del fold 1 dio 0,99270, 0,99294 y 0,99315; lo que no se movió es
la métrica de decisión. El recall del fold 5 que la figura 3 midió en vivo, 0,9978, es el mismo. Lo
que esa medición no cubre es la sensibilidad a los hiperparámetros vecinos, que es otra pregunta.

**El artefacto.** La configuración se reajustó sobre toda la referencia, las 5.787.030 filas con
4.477 fraudes, dentro del mismo `Pipeline` que el arnés evalúa: `evaluation.compose`, la misma
función con la que se ajusta cada fold. Es el artefacto único del proyecto. La celda siguiente lo
carga y comprueba que es el que registra el ADR 0010.

In [ ]:
ADR_0010_SHA256 = "41869ee0e5239ca66e0024a10bff4b727cf7ba0a6b6a8087a45da010de5d4ebd"
ADR_0010_BYTES = 512_224
pipeline = load_pipeline(ARTIFACT_PATH)
digest = file_sha256(ARTIFACT_PATH)
assert digest == ADR_0010_SHA256
assert ARTIFACT_PATH.stat().st_size == ADR_0010_BYTES
model_params = pipeline.named_steps["model"].get_params()
assert {name: model_params[name] for name in CHOSEN_PARAMS} == dict(CHOSEN_PARAMS)
preprocess = pipeline.named_steps["preprocess"]
received = list(pipeline[:-1].get_feature_names_out())
print(f"{ARTIFACT_PATH.name}: {ARTIFACT_PATH.stat().st_size} bytes, sha256 {digest}")
print(f"Pasos: {list(pipeline.named_steps)}; preprocesamiento: {list(preprocess.named_steps)}")
print(f"Columnas crudas que lee (registry.MODEL_INPUT_COLUMNS): {list(MODEL_INPUT_COLUMNS)}")
print(f"El modelo recibe {len(received)} features: {received}")

El sha256 y el tamaño son los del ADR 0010, y los hiperparámetros del modelo cargado son los del
ensayo 5. Un solo objeto con dos pasos: el preprocesamiento, que deriva la bandera de la firma y la
hora y escala los montos, y el modelo. Lee columnas crudas y nada más. Está registrado en el Model
Registry de MLflow como `fraud-detection-pipeline`, **versión 1**, desde la corrida
`e1fb3e9b12ed4dfc9a025a751eb9ff04`, con firma de entrada, hiperparámetros y las etiquetas que
identifican los datos; la versión que vuelve del registro predice exactamente lo mismo que esta
copia (ADR 0010). El notebook 03, el monitoreo y la app cargan este objeto; ninguno reimplementa
una transformación. Y ninguna fila de producción participó en elegir nada de lo que contiene.

**Lo que este notebook no permite concluir:**

- que una búsqueda completa no encontraría algo mejor: se cortó en 8 de 150 ensayos;
- cuánto generaliza el modelo fuera de la referencia: toda la validación es interna al periodo de
  referencia, y producción no se tocó hasta la fase 4 (EVAL-011);
- por qué el estratificado dio más bajo, ni qué parámetro contiene la divergencia de LightGBM;
- nada sobre datos reales: la firma es del simulador, y en datos reales habría transferencias
  legítimas por el saldo completo.

## 7. Lo que hace la fase 3

La fase 3 explica **este** artefacto: lo carga y no reajusta nada, porque un modelo reconstruido
para la explicación explicaría otro objeto (ADR 0011). Tiene una ventaja que casi ningún proyecto
tiene: **la respuesta se conoce de antemano**. La firma del simulador es la verdad, así que antes
de producir una sola figura, SHAP tiene que encontrarla; si no la encuentra, el roto es el
explicador y no el modelo.

Después vienen las tres ordenaciones globales lado a lado —SHAP, permutación y la importancia
interna de LightGBM—, PDP e ICE sobre columnas crudas a través del artefacto completo, el umbral de
operación que elige la métrica de decisión de este notebook, y cuatro casos locales elegidos con un
criterio declarado antes de medir. Todo eso está en `notebooks/03_interpretability.ipynb`.